In [ ]:
import pandas as pd

file_path = '9_datasets.xlsx' 
excel_data = pd.ExcelFile(file_path)

df = pd.concat(
    [excel_data.parse(name).assign(Site=name) for name in excel_data.sheet_names],
    ignore_index=True
)

print(df.info())
print(df.head())

In [ ]:
categorical_columns = ['Overall Site ID', 'Gender', 'Race', 'Site']
for col in categorical_columns:
    df[col] = df[col].fillna(df[col].mode()[0])  

numeric_columns = [
    'Age (yrs)', 'Weight (lbs)', 'Height (in)', 
    'Resting DBP (mmHg)', 'Resting SBP (mmHg)', 'A1C (%)', 
    'BMI', '% Body Fat', 'Triglycerides (mg/dL)', 
    'Cholesterol (mg/dL)', 'LDL (mg/dL)', 'HDL (mg/dL)'
]
for col in numeric_columns:
    df[col] = df[col].fillna(df[col].median()) 

df = df.dropna(subset=["Visit ID", "Date of Round"])

In [ ]:
print("Cleaned Data Overview:")
print(df.info())
print(df.head())

In [ ]:
import statsmodels.formula.api as smf

df['Visit ID'] = df['Visit ID'].astype(str)

df.rename(columns={'Age (yrs)': 'Age_yrs'}, inplace=True)
model_bmi_age = smf.mixedlm("BMI ~ Age_yrs", data=df, groups=df["Visit ID"])

result_bmi_age = model_bmi_age.fit()

print("Linear Mixed Model for BMI ~ Age (with random effect for Visit ID):")
print(result_bmi_age.summary())

In [ ]:
df['Visit ID'] = df['Visit ID'].astype(str)
df.rename(columns={'Age (yrs)': 'Age_yrs'}, inplace=True)

df.rename(columns={
    'Resting DBP (mmHg)': 'Resting_DBP',
    'Resting SBP (mmHg)': 'Resting_SBP',
    'A1C (%)': 'A1C',
    '% Body Fat': 'Body_Fat',
    'Triglycerides (mg/dL)': 'Triglycerides',
    'Cholesterol (mg/dL)': 'Cholesterol',
    'LDL (mg/dL)': 'LDL',
    'HDL (mg/dL)': 'HDL'
}, inplace=True)

health_indicators = [
    'Resting_DBP', 'Resting_SBP', 'A1C', 'BMI', 
    'Body_Fat', 'Triglycerides', 'Cholesterol', 
    'LDL', 'HDL'
]

for indicator in health_indicators:

    model = smf.mixedlm(f"{indicator} ~ Age_yrs", data=df, groups=df["Visit ID"])
    result = model.fit()
    print(f"Linear Mixed Model for {indicator} ~ Age (with random effect for Visit ID):")
    print(result.summary())
    print("\n" + "-"*100 + "\n")

In [ ]:
df = df.rename(columns={"Date of Round": "DateOfRound"})

try:
    model_date = smf.mixedlm(f"{indicator} ~ DateOfRound", data=df, groups=df["Visit ID"])
    result_date = model_date.fit()
    print(f"Linear Mixed Model for {indicator} ~ DateOfRound (with random effect for Visit ID):")
    print(result_date.summary())
except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:
df.columns = df.columns.str.strip()

print(df.columns)
df['Date_of_Round'] = pd.to_datetime(df['DateOfRound'])

def assign_quarter(month):
    if month in [1, 2, 3]:
        return 1 
    elif month in [4, 5, 6]:
        return 2 
    elif month in [7, 8, 9]:
        return 3 
    else:
        return 4

df['Quarter'] = df['DateOfRound'].dt.month.apply(assign_quarter)
df['Quarter'] = df['Quarter'].astype('category')

health_indicators = [
    'Resting_DBP', 'Resting_SBP', 'A1C', 'BMI', 
    'Body_Fat', 'Triglycerides', 'Cholesterol', 
    'LDL', 'HDL'
]

for indicator in health_indicators:
    model = smf.mixedlm(f"{indicator} ~ Quarter", df, groups=df["Visit_ID"]).fit()
    print(f"{indicator} Model Summary")
    print(model.summary())
    print("\n" + "="*50 + "\n")  

In [ ]:
df['Race'] = df['Race'].astype('category')

health_indicators = [
    'Resting_DBP', 'Resting_SBP', 'A1C', 'BMI', 
    'Body_Fat', 'Triglycerides', 'Cholesterol', 
    'LDL', 'HDL'
]

for indicator in health_indicators:

    model = smf.mixedlm(f"{indicator} ~ Race", data=df, groups=df["Visit_ID"])
    result = model.fit()
    print(f"Linear Mixed Model for {indicator} ~ Race (with random effect for Visit ID):")
    print(result.summary())
    print("\n" + "-"*100 + "\n")